# Detecção de Fraudes em Cartões de Crédito

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_curve, roc_auc_score, precision_recall_curve
from xgboost import XGBClassifier
import shap

In [ ]:
df = pd.read_csv("creditcard.csv")
df.head()

In [ ]:
print(df["Class"].value_counts())
print(df["Class"].value_counts(normalize=True)*100)

In [ ]:
df["LogAmount"] = np.log1p(df["Amount"])
X=df.drop("Class",axis=1)
y=df["Class"]

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [ ]:
lr=LogisticRegression(class_weight="balanced",max_iter=1000)
lr.fit(X_train_scaled,y_train)
print(classification_report(y_test,lr.predict(X_test_scaled)))

In [ ]:
rf=RandomForestClassifier(n_estimators=300,class_weight="balanced",random_state=42)
rf.fit(X_train,y_train)
print(classification_report(y_test,rf.predict(X_test)))

In [ ]:
scale=sum(y_train==0)/sum(y_train==1)
xgb=XGBClassifier(scale_pos_weight=scale,n_estimators=300,max_depth=5,learning_rate=0.05,eval_metric="logloss")
xgb.fit(X_train,y_train)
print(classification_report(y_test,xgb.predict(X_test)))

In [ ]:
probs=xgb.predict_proba(X_test)[:,1]
precision,recall,_=precision_recall_curve(y_test,probs)
plt.plot(recall,precision)
plt.show()

In [ ]:
explainer=shap.TreeExplainer(xgb)
sample=X_test.sample(min(2000,len(X_test)),random_state=42)
shap_values=explainer.shap_values(sample)
shap.summary_plot(shap_values,sample)